# CALPHAD workflow with ATAT and machine-learned potentials

This notebook fits CALPHAD databases with `CalphadMaker` and draws the phase
diagrams. We compute Ni-Re, Co-Ni and Cr-V, the three binaries of Zhu et al.
(npj Comput. Mater. 11, 340 (2025)), with four machine-learned potentials. No
DFT is needed. The [force field documentation](https://materialsproject.github.io/atomate2/user/codes/forcefields.html#calphad)
describes the workflow.

This workflow is new and has not been tested widely. It might still change in
future versions.

| Part | Content | Needs | Time |
|---|---|---|---|
| A. Run the workflows | Ni-Re step by step, Co-Ni and Cr-V, the ATAT fits | GPUs, the model files and ATAT | about 3.5 h for Ni-Re with MACE on one A100 GPU |
| B. Our results | All phase diagrams and tables, drawn from the TDB files of our runs | atomate2 and pycalphad | about 2 min |

To see the results, run only Part B. Select its first code cell and use "Run
Selected Cell and All Below". "Run All" also starts the workflow runs of Part A.

Contents:

- Background
- Installation
- Part A. Run the workflows
  - A1. Ni-Re step by step
  - A2. Co-Ni and Cr-V
  - A3. Short range order
  - A4. Saving the TDB files for Part B
- Part B. Our results
  - B1. Ni-Re
  - B2. Co-Ni
  - B3. Cr-V
  - B4. Short range order
  - B5. Liquid settings
  - B6. Comparison with PhaseForge
- Recommended settings
- Known limitations

## Background

The workflow follows the `sqs2tdb` approach of ATAT (van de Walle et al.,
Calphad 58, 70 (2017)).

**Step 1: SQS.** `sqs2tdb` copies the special quasirandom structures (SQS) of
each lattice from the ATAT database. At level 2 these are the pure elements and
the compositions with 25, 50 and 75 at% of each element.

**Step 2: Solids.** Each solid SQS is relaxed with the force field, including the
cell.

**Step 3: Vibrational entropy.** The atoms of each relaxed solid SQS are relaxed
again at fixed cell. The displacements run in a diagonal supercell with all
lattice vectors at least 20 Å long. The harmonic entropy is phonopy's sum over a
q-point mesh at 3000 K, without the three acoustic modes at Gamma.

**Step 4: Liquids.** Each liquid SQS is repeated three times along each lattice
vector, which gives 864 atoms. It is melted at `melt_temperature` and then run
at `liquid_temperature`, both NPT at zero pressure. The mean potential energy of
the second run is the energy of the liquid.

**Step 5: Fit.** `sqs2tdb` fits the mixing energies and vibrational entropies of
each lattice and writes one TDB file, which pycalphad can read.

The mixing energy of a phase of A and B is a Redlich-Kister sum,
x<sub>A</sub> x<sub>B</sub> [L0 + L1 (x<sub>A</sub> − x<sub>B</sub>) + ...].
The default terms fit L0 and L1. For a solid each term is a + b T. The mixing
energy gives a, and the excess vibrational entropy gives b. The liquid terms
have no T part. For lattices in the SGTE database, `sqs2tdb` takes the free
energies of the pure elements from SGTE (Dinsdale, Calphad 15, 317 (1991)), so
the melting points of the pure elements are the experimental ones.

## Installation

Part A needs the `ase` and `phonons` extras, MACE and pycalphad:

```
pip install 'atomate2[ase,phonons]' 'mace-torch>=0.3.16' pycalphad
```

GRACE needs `tensorpotential`, which brings TensorFlow. PyTorch and TensorFlow
each install their own CUDA libraries, so we ran GRACE in a separate
environment. Set `GRACE_CACHE` to keep its models outside `~/.cache/grace`.

```
pip install 'atomate2[ase,phonons]' tensorpotential pycalphad
export TF_USE_LEGACY_KERAS=1
```

ATAT is not a Python package. The workflow needs the `sqs2tdb` script and three
ATAT programs, which build in a few seconds. The ATAT `src` folder must be on
the `PATH` of the shell that starts Jupyter.

```
curl -sSLO https://axelvandewalle.github.io/www-avdw/atat/atat3_52.tar.gz
tar xzf atat3_52.tar.gz
make -C atat/src cellcvrt nntouch lsfit
echo "set atatdir=$PWD/atat" > ~/.atat.rc
export PATH=$PWD/atat/src:$PATH
```

Part B needs only atomate2 and pycalphad.

## Part A. Run the workflows

### Potentials

We use four foundation models.

- MACE-OMAT-0-medium. `mace_mp(model="medium-omat-0")` downloads and caches it.
- MACE-MATPES-PBE-0 and MACE-MATPES-r2SCAN-0. These are MACE-OMAT-0 fine-tuned
  on MatPES at the PBE and r2SCAN levels. Download
  [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  and [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model).
- GRACE-2L-OMAT, a two-layer GRACE model fitted on OMat24. `grace_fm` of
  `tensorpotential` downloads it by name. atomate2 has no name for GRACE, so the
  calculator is given as a dictionary that points to `grace_fm`.

The next cell checks that ATAT is on the `PATH` and sets the potentials. Replace
the two MACE paths with your files. The liquid MD needs a GPU. MACE runs in
`float64`, since the mixing energies are small differences of large total
energies. GRACE runs in `float64` by default.

In [ ]:
import shutil

for program in ("sqs2tdb", "cellcvrt", "nntouch", "lsfit"):
    if shutil.which(program) is None:
        raise OSError(f"{program} is not on the PATH")

MACE = {"device": "cuda", "default_dtype": "float64"}
GRACE = {
    "@module": "tensorpotential.calculator.foundation_models",
    "@callable": "grace_fm",
}

# Replace the two paths with the files you downloaded.
FORCE_FIELDS = {
    "MACE-OMAT-0-medium": ("MACE-MP-0", {"model": "medium-omat-0", **MACE}),
    "MACE-MATPES-PBE-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-pbe-omat-ft.model", **MACE},
    ),
    "MACE-MATPES-r2SCAN-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-r2scan-omat-ft.model", **MACE},
    ),
    "GRACE-2L-OMAT": (GRACE, {"model": "GRACE-2L-OMAT"}),
}

### A1. Ni-Re step by step

#### Lattices and terms

Ni is stable in FCC_A1 and Re in HCP_A3. We fit these two lattices, the liquid
and two ordered lattices of the ATAT database, NI3SN_D019 and NI4MO_D1A. The
ordered lattices test whether a force field makes an ordered phase stable.

`terms` gives the lines of the `sqs2tdb` `terms.in` file. Each line is an
`order,level` pair. Order 1 gives the end members and order 2 the mixing. The
ordered lattices have two sublattices, so their lines have one pair per
sublattice. `sqs2tdb` applies each line to every sublattice.

At level 2 the ordered lattices have too few SQS for L1, so they take L0 only.
With L1 for FCC_A1, HCP_A3 and LIQUID and L0 for the ordered lattices,
NI3SN_D019 was stable up to about 3000 K with MACE-OMAT-0-medium and
GRACE-2L-OMAT. We therefore fit L0 only for every lattice of Ni-Re. Adding L1
does not change L0, since the SQS compositions are symmetric around 50 at%.

#### Liquid temperatures

The liquid temperatures have no defaults. The melt at 4500 K makes every
composition melt within 10 ps. All compositions then run at 2800 K, below the
melting point of Re (3458 K in SGTE). The melted cells stay liquid during the
20 ps run. B5 explains this choice.

In [ ]:
from atomate2.forcefields.flows.calphad import CalphadMaker

TERMS = {
    "FCC_A1": ["1,0", "2,0"],
    "HCP_A3": ["1,0", "2,0"],
    "NI3SN_D019": ["1,0:1,0", "2,0:1,0"],
    "NI4MO_D1A": ["1,0:1,0", "2,0:1,0"],
    "LIQUID": ["1,0", "2,0"],
}


def get_maker(model: str, **kwargs) -> CalphadMaker:
    """Return a CALPHAD maker with one of the potentials."""
    force_field, calculator_kwargs = FORCE_FIELDS[model]
    return CalphadMaker.from_force_field_name(
        force_field, calculator_kwargs=calculator_kwargs, **kwargs
    )


NI_RE = {
    "melt_temperature": 4500,
    "liquid_temperature": 2800,
    "lattices": list(TERMS),
    "terms": TERMS,
}
flow = get_maker("MACE-OMAT-0-medium", **NI_RE).make(["Ni", "Re"])
[job.name for job in flow.jobs]

#### Run

The flow computes 29 SQS, 24 solids and 5 liquids, and the phonons of each
solid. On one A100 GPU the solid relaxations took a few minutes together. Each
liquid run took 40 to 46 min with MACE and about 22 min with GRACE-2L-OMAT.
`run_locally` runs the liquids one after the other, about 3.5 h with MACE, so
the next cell is off by default. A workflow manager can run the five liquids at
once on five GPUs.
`create_folders=True` gives each job its own folder, and the fit job runs
`sqs2tdb` in its folder.

In [ ]:
from jobflow import run_locally

RUN_WORKFLOW = False
if RUN_WORKFLOW:
    responses = run_locally(flow, create_folders=True, ensure_success=True)
    calphad_doc = responses[flow.output.uuid][1].output

#### Checking the output

The output is a `CalphadDoc`. It holds the TDB file in `tdb` and the energy of
each SQS in `calculations`. Each solid also has its `vibrational_entropy` in k_B
per SQS cell. We check four fields.

- `relaxation_strain` of a solid measures how far the relaxed cell is from the
  ideal lattice. The ATAT `checkrelax` help calls values above 0.1 too large.
- `is_force_converged` says whether the relaxation reached `fmax` within its
  steps.
- `imaginary_fraction` of a solid is the part of its phonon frequencies on the
  q-point mesh below -0.05 THz. If it is above `max_imaginary_fraction` (0.03 by
  default) for one SQS of a lattice, that lattice is fitted to the energies only,
  with a warning.
- `mean_squared_displacement` of a liquid keeps growing during the run. In a
  crystal it stays well below 1 Å².

In [ ]:
import pandas as pd

checks = None
if RUN_WORKFLOW:
    checks = pd.DataFrame(
        [
            calc.model_dump(exclude={"structure", "dir_name"})
            for calc in calphad_doc.calculations
        ]
    )
checks

The next table gives these checks for our first run at 2800 K with each
potential. The solids were relaxed with `fmax` = 0.001 eV/Å and at most 500
steps. The default is now 0.01 eV/Å and 1000 steps. The last five columns are the
mean squared displacement of the liquid in Å², by at% Re.

| Potential | Largest strain | Not converged | 0 | 25 | 50 | 75 | 100 |
|---|---|---|---|---|---|---|---|
| MACE-OMAT-0-medium | 0.033 | 2 of 24 | 118.0 | 76.6 | 47.2 | 31.2 | 16.8 |
| MACE-MATPES-PBE-0 | 0.028 | 1 of 24 | 119.1 | 75.1 | 45.5 | 28.8 | 16.1 |
| MACE-MATPES-r2SCAN-0 | 0.026 | 3 of 24 | 117.0 | 66.1 | 41.3 | 24.5 | 14.3 |
| GRACE-2L-OMAT | 0.038 | 4 of 24 | 109.3 | 74.5 | 49.6 | 27.4 | 15.3 |

All strains are far below 0.1. No solid SQS had an imaginary frequency on the
mesh. The MACE relaxations repeated with 2000 steps gave the same energies. Every
liquid has a mean squared displacement far above 1 Å², so every liquid SQS has
melted.

#### Fitting the TDB file

We refit the TDB files from the results of our runs in
`tests/test_data/common/calphad/`. These files hold the total energy in eV of
each SQS cell and the vibrational entropy in k_B of each solid SQS cell. The
liquid energies are per 32-atom SQS, from the first of our MD runs. `fit_tdb` is
the last job of the flow, and `.original` calls its function directly. It writes
the `sqs2tdb` folders into the current folder, so each fit runs in its own
folder.

In [ ]:
import contextlib
import json
from pathlib import Path

from mock_vasp import TEST_DIR

from atomate2.common.jobs.calphad import fit_tdb

DATA_DIR = TEST_DIR / "common" / "calphad"


def fit_tdbs(
    system: str,
    elements: list[str],
    terms: dict,
    short_range_order: bool = False,
    extra: dict | None = None,
) -> dict[str, str]:
    """Fit the TDB file of each potential from the results of our runs.

    extra adds the calculations of other lattices for the potentials it holds.
    """
    energies = json.loads((DATA_DIR / f"{system}_energies.json").read_text())
    folder = (
        system + ("_sro" if short_range_order else "") + ("_extra" if extra else "")
    )
    tdbs = {}
    for model, calculations in energies.items():
        if extra is not None and model not in extra:
            continue
        fit_dir = Path("calphad_fits", folder, model)
        fit_dir.mkdir(parents=True, exist_ok=True)
        with contextlib.chdir(fit_dir):
            tdbs[model] = fit_tdb.original(
                elements,
                2,
                terms,
                calculations + (extra or {}).get(model, []),
                short_range_order=short_range_order,
            ).tdb
    return tdbs


TDBS = {"ni_re": fit_tdbs("ni_re", ["Ni", "Re"], TERMS)}

### A2. Co-Ni and Cr-V

Co-Ni uses the same steps as Ni-Re. Ni is stable in FCC_A1 and Co in HCP_A3. We
fit these two lattices and the liquid with the default terms, L0 and L1. The
liquid is melted at 3000 K and run at 2000 K, about 230 K above the melting
points of both elements.

Cr and V are both stable in BCC_A2. BCC_A2 is not in the default lattices, so we
add it. Its default terms are L0 and L1. The liquid is melted at 3500 K and run
at 2400 K, about 220 K above the melting points of both elements. The solids of
both binaries were relaxed with the default settings.

`RUN_ALL = True` runs the eight workflows. The cell then fits both binaries from
our results.

In [ ]:
CO_NI = {
    "melt_temperature": 3000,
    "liquid_temperature": 2000,
    "lattices": ["FCC_A1", "HCP_A3", "LIQUID"],
}
CR_V = {
    "melt_temperature": 3500,
    "liquid_temperature": 2400,
    "lattices": ["BCC_A2", "LIQUID"],
}
co_ni_maker = get_maker("MACE-OMAT-0-medium", **CO_NI)
cr_v_maker = get_maker("MACE-OMAT-0-medium", **CR_V)
CO_NI_TERMS = {lattice: co_ni_maker.terms[lattice] for lattice in co_ni_maker.lattices}
CR_V_TERMS = {lattice: cr_v_maker.terms[lattice] for lattice in cr_v_maker.lattices}

RUN_ALL = False
if RUN_ALL:
    for model in FORCE_FIELDS:
        for elements, settings in ((["Co", "Ni"], CO_NI), (["Cr", "V"], CR_V)):
            flow = get_maker(model, **settings).make(elements)
            run_locally(flow, create_folders=True, ensure_success=True)

TDBS["co_ni"] = fit_tdbs("co_ni", ["Co", "Ni"], CO_NI_TERMS)
TDBS["cr_v"] = fit_tdbs("cr_v", ["Cr", "V"], CR_V_TERMS)

#### Lattices that are not stable

Zhu et al. fit FCC_A1, BCC_A2 and HCP_A3 for every binary. We also relaxed the
BCC_A2 SQS of Co-Ni and the FCC_A1 and HCP_A3 SQS of Cr-V, which are not stable
for these elements. The next cell adds their energies to our fits for
MACE-OMAT-0-medium. B6 shows the result.

In [ ]:
EXTRA = json.loads((DATA_DIR / "extra_lattices_energies.json").read_text())
EXTRA_TERMS = ["1,0", "2,1"]
TDBS["co_ni_extra"] = fit_tdbs(
    "co_ni",
    ["Co", "Ni"],
    {**CO_NI_TERMS, "BCC_A2": EXTRA_TERMS},
    extra=EXTRA["co_ni"],
)
TDBS["cr_v_extra"] = fit_tdbs(
    "cr_v",
    ["Cr", "V"],
    {**CR_V_TERMS, "FCC_A1": EXTRA_TERMS, "HCP_A3": EXTRA_TERMS},
    extra=EXTRA["cr_v"],
)

### A3. Short range order

An SQS mimics a random alloy. In a real alloy the atoms order a little at short
range, which lowers the free energy of the solid, most at low temperature.
`short_range_order=True` adds the low-order cluster variation method (CVM)
approximation of `sqs2tdb -fit -sro`. It needs no new calculations. It uses the
energy part of L0 (and of L2, if fitted) and the number of nearest neighbours,
12 for FCC_A1 and HCP_A3 and 8 for BCC_A2. The correction goes to zero at high
temperature. It is only added to FCC_A1, BCC_A2, HCP_A3 and DIAMOND_A4, so the
liquid and the ordered lattices are unchanged. It is off by default.

In [ ]:
TDBS["ni_re_sro"] = fit_tdbs("ni_re", ["Ni", "Re"], TERMS, short_range_order=True)
TDBS["co_ni_sro"] = fit_tdbs("co_ni", ["Co", "Ni"], CO_NI_TERMS, short_range_order=True)
TDBS["cr_v_sro"] = fit_tdbs("cr_v", ["Cr", "V"], CR_V_TERMS, short_range_order=True)

### A4. Saving the TDB files for Part B

The next cell writes the TDB files of A1 to A3 to
`tests/test_data/common/calphad/calphad_tdbs.json.gz`, which Part B reads. To
draw your own fits, write them to another file and set `TDB_FILE` in Part B.

In [ ]:
from datetime import UTC, datetime
from importlib.metadata import version

from monty.serialization import dumpfn

TDB_FILE = "../tests/test_data/common/calphad/calphad_tdbs.json.gz"
dumpfn(
    {
        "provenance": {
            "energies": "tests/test_data/common/calphad/*_energies.json",
            "potentials": list(FORCE_FIELDS),
            "atomate2": version("atomate2"),
            "atat": "3.52",
            "date": datetime.now(UTC).date().isoformat(),
        },
        "tdbs": TDBS,
    },
    TDB_FILE,
)

## Part B. Our results

Part B draws all phase diagrams and tables from the TDB files of our runs. It
needs only atomate2 and pycalphad. The next cell loads the TDB files and prints
where they come from.

In [ ]:
import re
import warnings

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from monty.serialization import loadfn
from pycalphad import Database, binplot, equilibrium
from pycalphad import variables as v

# pycalphad warns about the type definitions of the SGTE parameters in each TDB
warnings.filterwarnings("ignore", message="The type definition character")

TDB_FILE = "../tests/test_data/common/calphad/calphad_tdbs.json.gz"
results = loadfn(TDB_FILE)
tdbs = results["tdbs"]
results["provenance"]

In [ ]:
# one colour per phase in every diagram, away from the green tie lines and the
# red three-phase lines that pycalphad draws
PHASE_COLORS = {
    "LIQUID": "#F4C800",
    "FCC_A1": "#00538A",
    "BCC_A2": "#704AA4",
    "HCP_A3": "#FF8E00",
    "NI3SN_D019": "#7BD1EC",
    "NI4MO_D1A": "#803E75",
}


def phase_legend(phases: list[str]) -> tuple[list, dict[str, str]]:
    """Return the legend handles and the colour of each phase for binplot.

    pycalphad names a second composition set of a phase as e.g. NI4MO_D1A#2.
    """
    colors = {phase: PHASE_COLORS[phase.split("#")[0]] for phase in phases}
    names = sorted({phase.split("#")[0] for phase in phases})
    handles = [mpatches.Patch(color=PHASE_COLORS[p], label=p) for p in names]
    return handles, colors


def plot_diagrams(
    tdbs: dict[str, str], elements: list[str], temperatures: tuple, line: float = 0
) -> None:
    """Draw the phase diagram of each TDB file, with a dotted line at line K."""
    species = [element.upper() for element in elements]
    conditions = {v.X(species[1]): (0, 1, 0.02), v.T: temperatures, v.P: 101325, v.N: 1}
    rows = (len(tdbs) + 1) // 2
    fig, axes = plt.subplots(
        rows, 2, figsize=(11, 4.5 * rows), sharex=True, sharey=True
    )
    phases = set()
    for ax, (title, tdb) in zip(axes.flat, tdbs.items(), strict=True):
        db = Database.from_string(tdb, fmt="tdb")
        phases |= set(db.phases)
        plot_kwargs = {"ax": ax, "legend_generator": phase_legend}
        binplot(db, species, sorted(db.phases), conditions, plot_kwargs=plot_kwargs)
        if line:
            ax.axhline(line, color="black", ls=":")
        ax.get_legend().remove()
        ax.set_title(title)
        ax.label_outer()
    fig.legend(handles=phase_legend(sorted(phases))[0], loc="center right")
    fig.tight_layout(rect=(0, 0, 0.85, 1))
    plt.show()


def plot_melting(
    tdbs: dict[str, str], elements: list[str], temperatures: tuple
) -> None:
    """Draw the liquidus (solid line) and the solidus (dashed) of each TDB file.

    The liquidus is the lowest temperature of the grid at which only the liquid
    is stable. The solidus is the highest temperature without liquid.
    """
    species = [element.upper() for element in elements]
    conditions = {
        v.X(species[1]): (0, 1.001, 0.02),
        v.T: temperatures,
        v.P: 101325,
        v.N: 1,
    }
    _, ax = plt.subplots(figsize=(7, 5))
    for model, tdb in tdbs.items():
        db = Database.from_string(tdb, fmt="tdb")
        eq = equilibrium(db, [*species, "VA"], sorted(db.phases), conditions)
        phases = eq.Phase.values.squeeze()
        liquid = (phases == "LIQUID").any(axis=-1)
        solid = ((phases != "LIQUID") & (phases != "")).any(axis=-1)
        temperature = eq.T.values
        liquidus = [temperature[~column].min() for column in solid.T]
        solidus = [temperature[~column].max() for column in liquid.T]
        x = eq[f"X_{species[1]}"].values
        (line,) = ax.plot(x, liquidus, label=model)
        ax.plot(x, solidus, ls="--", color=line.get_color())
    ax.set_xlabel(f"X({species[1]})")
    ax.set_ylabel("Temperature (K)")
    ax.legend()
    plt.show()


# a TDB parameter such as +1357.5-1.9061*T, or -3860.4 for the liquid
TERM = re.compile(r"([-+][\d.]+)(?:([-+][\d.]+)\*T)?")


def get_mixing_terms(
    tdbs: dict[str, str], lattices: list[str], level: int
) -> pd.DataFrame:
    """Return L0 to L<level> = a + b T of each lattice, read from the TDB files.

    a is in J/mol and b in J/(mol K). b is 0 for the liquid.
    """
    rows = {}
    for model, tdb in tdbs.items():
        rows[model] = {}
        for lattice in lattices:
            for order in range(level + 1):
                line = next(
                    line
                    for line in tdb.splitlines()
                    if f"L({lattice}," in line and f";{order})" in line
                )
                a, b = TERM.fullmatch(line.split()[3]).groups()
                rows[model][f"{lattice} L{order} a"] = float(a)
                rows[model][f"{lattice} L{order} b"] = float(b or 0)
    return pd.DataFrame(rows).T.round(2)

### B1. Ni-Re

The next cell draws the four diagrams and gives L0 = a + b T, with a in J/mol
and b in J/(mol K). The dotted line is the experimental peritectic at 1834 K.

In [ ]:
plot_diagrams(tdbs["ni_re"], ["Ni", "Re"], (1000, 4000, 25), line=1834)
get_mixing_terms(tdbs["ni_re"], ["FCC_A1", "HCP_A3", "LIQUID"], 0)

L0 in J/mol, as printed by the cell above:

| Potential | FCC_A1 | HCP_A3 | LIQUID |
|---|---|---|---|
| MACE-OMAT-0-medium | +7168 + 0.42 T | +15938 − 0.35 T | +366 |
| MACE-MATPES-PBE-0 | −2723 + 2.86 T | +2481 + 2.90 T | −17579 |
| MACE-MATPES-r2SCAN-0 | +3980 − 0.11 T | +14100 − 0.03 T | −13920 |
| GRACE-2L-OMAT | +1358 − 1.90 T | +892 + 3.36 T | −3860 |

A b of 1 J/(mol K) is an excess vibrational entropy of −0.03 k_B/atom at
50 at% Re, so the vibrational entropy changes FCC_A1 and HCP_A3 little. The b
values of the ordered lattices reach 56 J/(mol K) in size.

The temperatures below are from `pycalphad.equilibrium` on a grid with steps of
0.02 in composition and 5 K in temperature.

- MACE-OMAT-0-medium. At about 1890 K the liquid and HCP_A3 form NI4MO_D1A in a
  peritectic. Near Ni the liquid ends at about 1715 K, in a eutectic with FCC_A1
  and NI4MO_D1A.
- MACE-MATPES-PBE-0 and MACE-MATPES-r2SCAN-0. The liquid is stable far below the
  melting point of Ni. It ends near 30 at% Re at about 1585 K (PBE) and 1475 K
  (r2SCAN), where it freezes into NI4MO_D1A and, with PBE, also HCP_A3. Their
  liquid L0 is much more negative than L0 of FCC_A1.
- GRACE-2L-OMAT. At about 2150 K the liquid and HCP_A3 form NI4MO_D1A in a
  peritectic. Near Ni, FCC_A1 forms from the liquid and NI4MO_D1A at about
  1745 K.

In experiment Ni-Re has a peritectic L + HCP → FCC at 1834 K, with about 11 at%
Re in the liquid, 20 at% in FCC and 60 at% in HCP (Boettinger et al., Metall.
Mater. Trans. A (2019)). No ordered phase has been reported. MACE-OMAT-0-medium
comes closest, with its peritectic about 55 K above experiment, but it forms
NI4MO_D1A in place of FCC_A1. The peritectic of GRACE-2L-OMAT is about 315 K too
high. Without the vibrational entropy it was about 40 K too low.

All four potentials make ordered phases stable. NI4MO_D1A is stable up to 1890 K
(MACE-OMAT-0-medium), 1670 K (PBE), 1485 K (r2SCAN) and 2150 K (GRACE-2L-OMAT).
With PBE, NI3SN_D019 is stable up to 1530 K. The known limitations discuss why
the ordered lattices may be too stable.

### B2. Co-Ni

The next cell draws the four diagrams and gives L0 and L1.

In [ ]:
plot_diagrams(tdbs["co_ni"], ["Co", "Ni"], (300, 2200, 10))
get_mixing_terms(tdbs["co_ni"], ["FCC_A1", "HCP_A3", "LIQUID"], 1)

Near the melting points the two-phase region of the liquid and FCC_A1 is only a
few K wide. With GRACE-2L-OMAT `binplot` loses it between about 15 and 87 at% Ni.
The next cell finds the liquidus and the solidus on a grid with steps of 0.02 in
composition and 2 K in temperature.

In [ ]:
plot_melting(tdbs["co_ni"], ["Co", "Ni"], (1600, 1800, 2))

Mixing terms, as a in J/mol and b in J/(mol K):

| Potential | FCC_A1 L0 | FCC_A1 L1 | HCP_A3 L0 | HCP_A3 L1 | LIQUID L0 | LIQUID L1 |
|---|---|---|---|---|---|---|
| MACE-OMAT-0-medium | −3268, +3.30 | −914, +0.52 | −3761, +2.64 | −1351, +0.49 | +677 | −922 |
| MACE-MATPES-PBE-0 | −6690, +3.40 | +2747, +0.22 | −7454, +4.04 | +1872, +0.76 | −3868 | +1518 |
| MACE-MATPES-r2SCAN-0 | +1789, +0.25 | +3331, −0.63 | −198, +0.95 | +3569, −1.21 | +162 | +3790 |
| GRACE-2L-OMAT | −2760, +1.66 | +421, +0.12 | −3442, +2.67 | +227, +0.16 | +439 | +1382 |

In our runs the largest `relaxation_strain` was 0.011, and one or two of the 10
solid relaxations did not reach `fmax` within 1000 steps. No solid SQS had an
imaginary frequency. The mean squared displacement of the liquids at 2000 K was
36 to 58 Å².

In experiment Co-Ni is a continuous FCC solid solution, with the liquidus and
solidus close together between the melting points of Co (1768 K) and Ni
(1728 K). HCP is stable on the Co side at low temperature (Nishizawa and Ishida,
Bull. Alloy Phase Diagr. 4, 390 (1983)). All four potentials give the continuous
FCC_A1 solution and the HCP_A3 field on the Co side. With MACE-MATPES-r2SCAN-0,
FCC_A1 also splits into two phases below about 370 K.

- With GRACE-2L-OMAT the liquidus and the solidus fall from Co to Ni, as in
  experiment.
- With the three MACE potentials they meet at a shallow minimum, 30 to 65 K
  below the melting point of Ni.

The positive b of the FCC_A1 L0 terms makes the solid less stable at high
temperature. Without it, MACE-OMAT-0-medium, MACE-MATPES-PBE-0 and GRACE-2L-OMAT
gave a maximum up to about 80 K above the melting point of Co.

Temperature of the minimum in K. The last column is the mean and standard
deviation over ten liquid MD runs with different random seeds (B5).

| Potential | Our fit | Ten runs |
|---|---|---|
| MACE-OMAT-0-medium | 1698 at 40 at% Ni | 1692 ± 14 |
| MACE-MATPES-PBE-0 | 1664 at 40 at% Ni | 1650 ± 11 |
| MACE-MATPES-r2SCAN-0 | 1678 at 62 at% Ni | 1688 ± 12 |

### B3. Cr-V

The next cell draws the four diagrams and gives L0 and L1. The dotted line is the
experimental liquidus minimum at 2043 K.

In [ ]:
plot_diagrams(tdbs["cr_v"], ["Cr", "V"], (1000, 3000, 10), line=2043)
get_mixing_terms(tdbs["cr_v"], ["BCC_A2", "LIQUID"], 1)

Mixing terms, as a in J/mol and b in J/(mol K):

| Potential | BCC_A2 L0 | BCC_A2 L1 | LIQUID L0 | LIQUID L1 |
|---|---|---|---|---|
| MACE-OMAT-0-medium | −21643, +11.24 | +958, −3.51 | −9250 | +1427 |
| MACE-MATPES-PBE-0 | −26007, +13.47 | +309, −2.99 | −6867 | −758 |
| MACE-MATPES-r2SCAN-0 | −24127, +7.44 | −2749, −2.68 | +562 | +2441 |
| GRACE-2L-OMAT | −23221, +13.80 | −1986, −2.46 | −7433 | +575 |

In our runs all BCC_A2 relaxations converged, and the largest
`relaxation_strain` was 0.006. The largest `imaginary_fraction` was 0.013, for
pure V with MACE-OMAT-0-medium, below the default `max_imaginary_fraction` of
0.03. The mean squared displacement of the liquids at 2400 K was 46 to 95 Å².

In experiment Cr-V is a continuous BCC solid solution, with a liquidus minimum
at about 2043 K and 30 at% V ([Liquidus Projection of the Cr-Nb-V System](https://doi.org/10.1007/s11669-026-01228-4),
which follows Smith, Bailey and Carlson (1982) and Ghosh (2002)). All four
potentials give the continuous BCC_A2 solution. Without the vibrational entropy,
the solid and the liquid met at a maximum 300 to 700 K above the melting points.

The excess vibrational entropy of BCC_A2 at 50 at% V is −0.22 to −0.41 k_B/atom.
Near the melting points it makes the solid less stable by 4 to 7.6 kJ/mol. With
MACE-OMAT-0-medium, MACE-MATPES-PBE-0 and GRACE-2L-OMAT the maximum becomes a
minimum at 1894 to 1976 K, 70 to 150 K below experiment, but at 62 to 64 at% V.
With MACE-MATPES-r2SCAN-0 the liquid L0 is positive, and a maximum remains at
2432 K and 28 at% V.

Temperature of the minimum or maximum in K, on a grid with steps of 0.02 in
composition and 2 K in temperature. The last column is over ten liquid MD runs,
as for Co-Ni.

| Potential | Our fit | Ten runs |
|---|---|---|
| MACE-OMAT-0-medium (minimum) | 1922 at 64 at% V | 1923 ± 12 |
| MACE-MATPES-PBE-0 (minimum) | 1976 at 62 at% V | 1983 ± 15 |
| MACE-MATPES-r2SCAN-0 (maximum) | 2432 at 28 at% V | 2438 ± 23 |
| GRACE-2L-OMAT (minimum) | 1894 at 64 at% V | 1910 ± 14 |

The liquid MD at 2400 K is below the maximum of MACE-MATPES-r2SCAN-0, but the
cells stayed liquid during the 20 ps run.

### B4. Short range order

The next cell draws Cr-V with the short range order of A3.

In [ ]:
plot_diagrams(tdbs["cr_v_sro"], ["Cr", "V"], (1000, 3000, 10), line=2043)

Liquidus minimum or maximum in K, without and with short range order, on the
grids of B2 and B3:

| Potential | Co-Ni | Co-Ni with SRO | Cr-V | Cr-V with SRO |
|---|---|---|---|---|
| MACE-OMAT-0-medium | 1698 | 1698 | 1922 | 1936 |
| MACE-MATPES-PBE-0 | 1664 | 1666 | 1976 | 1996 |
| MACE-MATPES-r2SCAN-0 | 1678 | 1678 | 2432 (maximum) | 2446 (maximum) |
| GRACE-2L-OMAT | none | none | 1894 | 1910 |

Short range order raises the Cr-V liquidus by 14 to 20 K, and the minimum stays
at 62 to 64 at% V. In Co-Ni the liquidus moves by at most 2 K. In Ni-Re no
temperature of B1 moves by more than the 5 K step of the grid. These changes are
about as large as the scatter of one liquid run (B5).

### B5. Liquid settings

#### Size of the liquid cell

We ran each Ni-Re liquid SQS with three random seeds, set by `mb_velocity_seed`
of both liquid makers, at 3600 K. The next cell plots L0 of the liquid against
the size of the liquid cell. Open markers are the three runs. We ran
GRACE-2L-OMAT only at 864 atoms.

In [ ]:
# L0 of the Ni-Re liquid in J/mol at 3600 K, three seeds per cell size
L0_SIZE = {
    "MACE-OMAT-0-medium": {
        256: (-713, -1889, -1438),
        864: (-2892, -3776, -3475),
        2048: (-2980, -3145, -3171),
    },
    "MACE-MATPES-PBE-0": {
        256: (-18711, -15533, -18916),
        864: (-16195, -17356, -17872),
        2048: (-17794, -16912, -15894),
    },
    "MACE-MATPES-r2SCAN-0": {
        256: (-15507, -13384, -15209),
        864: (-14467, -16199, -16556),
        2048: (-15882, -15249, -14620),
    },
    "GRACE-2L-OMAT": {864: (-9181, -5745, -7033)},
}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (model, by_size) in zip(axes, L0_SIZE.items(), strict=True):
    sizes = np.array(list(by_size))
    values = np.array(list(by_size.values())) / 1000
    for size, row in zip(sizes, values, strict=True):
        ax.plot([size] * 3, row, "o", mfc="white", color="C0")
    ax.errorbar(
        sizes, values.mean(axis=1), values.std(axis=1, ddof=1), fmt="o-", capsize=3
    )
    ax.set_xscale("log")
    ax.set_xlim(180, 2900)
    ax.set_xticks([256, 864, 2048], ["256", "864", "2048"])
    ax.minorticks_off()
    ax.set_title(model)
    ax.set_xlabel("Atoms in the liquid cell")
axes[0].set_ylabel("L0 of LIQUID at 3600 K (kJ/mol)")
fig.tight_layout()
plt.show()

Mean and standard deviation of L0 of the liquid in J/mol:

| Potential | 256 atoms | 864 atoms | 2048 atoms |
|---|---|---|---|
| MACE-OMAT-0-medium | −1347 ± 593 | −3381 ± 449 | −3098 ± 103 |
| MACE-MATPES-PBE-0 | −17720 ± 1897 | −17141 ± 859 | −16867 ± 951 |
| MACE-MATPES-r2SCAN-0 | −14700 ± 1149 | −15741 ± 1117 | −15250 ± 631 |
| GRACE-2L-OMAT | | −7320 ± 1736 | |

From 864 to 2048 atoms the mean changes by 270 to 490 J/mol, within the scatter
between runs. From 256 to 864 atoms it changes by 2030 J/mol with
MACE-OMAT-0-medium. The workflow therefore uses 864 atoms by default. On one A100
GPU a MACE liquid run took about 15 min at 256 atoms and 100 min at 2048 atoms.

#### Number of liquid runs

We ran each liquid SQS of the three binaries ten times with different random
seeds for 20 ps at the liquid temperatures of A1 and A2, and once for 100 ps.
Scatter of one 20 ps run, as one standard deviation:

| System | Liquid L0 (J/mol) | Liquidus maximum or minimum (K) | Liquid energy (meV/atom) |
|---|---|---|---|
| Ni-Re | 710 to 1200 | | 2.0 to 2.4 |
| Co-Ni | 430 to 560 | 11 to 14 | 1.3 to 1.4 |
| Cr-V | 630 to 840 | 12 to 23 | 1.3 to 1.5 |

The liquidus column leaves out GRACE-2L-OMAT for Co-Ni, which has no maximum or
minimum. One run never turned a maximum into a minimum or back. The 100 ps runs
showed no drift of the energy beyond this scatter, and all cells stayed liquid.

The workflow stores the standard error of each liquid energy in
`energy_standard_error`, from five blocks of the liquid MD after the
equilibration frames. In our runs it was 1.1 to 2.2 meV/atom, close to the
scatter between seeds. Propagated through the fit, it gave the scatter of L0
between runs to within about 30%.

#### Temperature of the liquid MD

The next cell plots L0 of the Ni-Re liquid at 864 atoms against
`liquid_temperature`, as the mean and standard deviation of three runs.

In [ ]:
# L0 of the Ni-Re liquid in J/mol at 864 atoms, three seeds per temperature
L0_T = {
    "MACE-OMAT-0-medium": {2800: (366, 235, -407), 3600: (-2892, -3776, -3475)},
    "MACE-MATPES-PBE-0": {
        2800: (-17579, -14247, -16550),
        3600: (-16195, -17356, -17872),
    },
    "MACE-MATPES-r2SCAN-0": {
        2800: (-13920, -14535, -13545),
        3600: (-14467, -16199, -16556),
    },
    "GRACE-2L-OMAT": {
        2800: (-3860, -4207, -6013),
        3200: (-5646, -5580, -6741),
        3600: (-9181, -5745, -7033),
        4000: (-7330, -10447, -9634),
    },
}

fig, ax = plt.subplots(figsize=(7, 4.5))
for model, by_temperature in L0_T.items():
    temperature = np.array(list(by_temperature))
    values = np.array(list(by_temperature.values())) / 1000
    ax.errorbar(
        temperature,
        values.mean(axis=1),
        values.std(axis=1, ddof=1),
        fmt="o-",
        capsize=3,
        label=model,
    )
ax.set_xticks([2800, 3200, 3600, 4000])
ax.set_xlabel("liquid_temperature (K)")
ax.set_ylabel("L0 of LIQUID, 864 atoms (kJ/mol)")
ax.legend()
plt.show()

Mean and standard deviation of L0 of the liquid in J/mol:

| Potential | 2800 K | 3200 K | 3600 K | 4000 K |
|---|---|---|---|---|
| MACE-OMAT-0-medium | +65 ± 414 | | −3381 ± 449 | |
| MACE-MATPES-PBE-0 | −16125 ± 1706 | | −17141 ± 859 | |
| MACE-MATPES-r2SCAN-0 | −14000 ± 500 | | −15741 ± 1117 | |
| GRACE-2L-OMAT | −4693 ± 1156 | −5989 ± 652 | −7320 ± 1736 | −9137 ± 1617 |

With GRACE-2L-OMAT, L0 changes by −3.7 ± 0.8 J/mol/K. With the MACE potentials it
rises by 1.0 to 3.4 kJ/mol from 3600 to 2800 K. The diagram is decided near the
liquidus, between 1728 and 3458 K, so `liquid_temperature` should be as low as
possible while every composition stays liquid. In all runs at 2800 K the energy
stayed flat after the first 5 ps and the mean squared displacement kept growing.

All compositions must run at the same temperature. Runs at 50 K above the mean
melting point of the elements of each composition, from 1778 K for Ni to 3509 K
for Re, gave L0 of the liquid near −12 kJ/mol with GRACE-2L-OMAT. The potential
energy of the liquid rises by 0.171 meV/atom/K for Ni and 0.234 meV/atom/K for
Re, which adds about −10 kJ/mol to L0 over 1778 to 3509 K.

### B6. Comparison with PhaseForge

Zhu et al. computed Ni-Re with the same `sqs2tdb` approach in their PhaseForge
code and with GRACE-2L-OMAT. Their peritectic is at 1904 K, 70 K above
experiment. We ran PhaseForge with GRACE-2L-OMAT and its default settings and
got their diagram. Its solid mixing terms have no T part and agree with our a
parts to 0.1 J/mol. Its L0 of the liquid is +3284 J/mol, against −3860 J/mol here.

PhaseForge runs each 32-atom liquid SQS for 2 ps in NVT, at 50 K above the mean
melting point of its elements. In our PhaseForge run the mean squared
displacement of all five cells was 0.05 to 0.11 Å². The cells stayed
crystalline, so their L0 of the liquid is a solid mixing energy. Zhu et al. note
that their agreement with experiment may come from a cancellation of errors.

#### Lattices that are not stable

Zhu et al. fit FCC_A1, BCC_A2 and HCP_A3 for every binary. Their Co-Ni diagram
has a BCC_A2 field near 50 at% Ni, which they note is not seen in experiment.
Their Cr-V diagram has an HCP_A3 field and a liquidus maximum near 3200 K. For
the lattices that are not stable for the elements:

- With three of the four potentials, the mixed BCC_A2 SQS of Co-Ni relax away
  from BCC, with a `relaxation_strain` of 0.12 to 0.17.
- With all four potentials, the mixed FCC_A1 and HCP_A3 SQS of Cr-V relax into
  BCC, with a `relaxation_strain` of 0.17 to 0.41.
- The phonons of the pure elements on these lattices have imaginary modes, up to
  5% of the modes for BCC Co and Ni and up to 26% for FCC and HCP Cr and V.

The next cell draws our fits of B2 and B3 for MACE-OMAT-0-medium next to the fits
with the energies of these lattices added (A2).

In [ ]:
model = "MACE-OMAT-0-medium"
plot_diagrams(
    {"Co-Ni": tdbs["co_ni"][model], "Co-Ni with BCC_A2": tdbs["co_ni_extra"][model]},
    ["Co", "Ni"],
    (300, 2600, 10),
)
plot_diagrams(
    {
        "Cr-V": tdbs["cr_v"][model],
        "Cr-V with FCC_A1 and HCP_A3": tdbs["cr_v_extra"][model],
    },
    ["Cr", "V"],
    (1000, 4000, 25),
)

The BCC_A2 field of Co-Ni then reaches about 2220 K. In Cr-V, HCP_A3 alone is
stable in the middle up to at least 4000 K. Both fields come from SQS that are no
longer on their lattice. We therefore leave these lattices out, and the fit job
warns when a `relaxation_strain` is above 0.1.

## Recommended settings

These settings worked for the three binaries with all four potentials.

- Use the default liquid cell of 864 atoms.
- Set `melt_temperature` high enough that every composition melts: 4500 K for
  Ni-Re, 3000 K for Co-Ni and 3500 K for Cr-V.
- Run all compositions at one `liquid_temperature`, as low as possible while every
  composition stays liquid: 2800 K for Ni-Re, 2000 K for Co-Ni and 2400 K for Cr-V.
- Keep the phonon step, which is on by default. With three of the four
  potentials it turned the Cr-V liquidus maximum into a minimum, as in experiment.
- Check that `relaxation_strain` of each solid is below 0.1, that
  `imaginary_fraction` of each solid is close to 0 and that
  `mean_squared_displacement` of each liquid is well above 1 Å².
- One liquid run per SQS was enough for these binaries. `energy_standard_error`
  shows the scatter of a single run.

## Known limitations

- The vibrational entropy is harmonic, from the relaxed SQS at 0 K, and its value
  at 3000 K is used at all temperatures. Anharmonic effects and thermal expansion
  are left out.
- The short range order of A3 is a low-order approximation from L0 only. A cluster
  expansion fitted to many ordered structures would describe it better.
- The liquid L0 is the excess energy at `liquid_temperature`, used at all
  temperatures. L0 changes with temperature, so the liquid has an excess heat
  capacity, which the fit leaves out. It would need free energies, for example
  from thermodynamic integration.
- The Ni side of the Ni-Re diagram is decided near 1800 K, far below 2800 K. A
  linear extrapolation of the mean GRACE-2L-OMAT L0 of three runs gives about
  −1 kJ/mol at 1800 K, against −4.7 kJ/mol at 2800 K.
- An ordered lattice takes the energies and vibrational entropies of its pure
  element end members from the force field, while FCC_A1 and HCP_A3 take them
  from SGTE. Pure Re on FCC sites can then have a lower free energy in NI4MO_D1A
  than in FCC_A1. This may help the ordered phases in Ni-Re.
- The mixing energies and vibrational entropies come from the force fields. A
  comparison with DFT for the same SQS would show how much of the difference to
  experiment comes from the force fields.